# 02 · Base models: custom CNN, VGG16, ResNet50, MobileNetV2

Phase 2 of the project, row A of the comparison table: the models learn from the real training images only.
This notebook:

1. loads the split from notebook 01 and the images, resized to 224 × 224 the same way the web app does it,
2. trains four models with the same settings: a small custom CNN (from random weights), and VGG16, ResNet50 and
   MobileNetV2 (from ImageNet weights, all layers trained),
3. keeps each model's best epoch, judged on the validation split,
4. measures accuracy, precision, recall and F1 per class, macro-F1, the confusion matrix, model size and prediction time,
5. saves the models, and MobileNetV2 also as `models/model.pt`, the file the web app loads.

**Two runs.** Set `FINAL` in the first code cell.

| `FINAL` | Seeds | Scores measured on | Time on a T4 GPU (estimate) | What it is for |
|---|---|---|---|---|
| `False` (trial) | 1 | validation split | about 1–2 hours | check that all four models train well; a first real model for the app |
| `True` (final) | 3 | test split | about 3–5 hours | the numbers for the report: mean ± standard deviation of the 3 seeds |

The test images are only loaded in the final run, so no choice is ever made by looking at them.

**How to run it on Kaggle**

- **Add Input** → search `plantvillage-dataset` (by *abdallahalidev*) → **Add**.
- **Add Input** → **Your Work** → **Notebooks** → your saved notebook 01 → **Add**. This gives `splits.csv`.
- **Settings**: Accelerator **GPU T4 x2** (the notebook uses one of the two GPUs). **Internet on**: it downloads the
  ImageNet weights (Kaggle asks for phone verification the first time).
- **Save Version → Save & Run All (Commit)**. The commit runs in the background, so you can close the browser. When it
  has finished, open the version's **Output** tab and go through the checklist at the end of this notebook.
- First commit with `FINAL = False`. If the checklist looks fine, set `FINAL = True` and commit again.
- Download `results/` and `models/model.pt` (about 9 MB) into the project folder on your laptop. The other files in
  `models/` (VGG16 alone is about 530 MB) can stay on Kaggle: notebooks 03 and 04 read them from this notebook's output.

In [ ]:
import os, json, math, time, random, shutil, hashlib, warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from PIL import Image
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import torch
from torch import nn
from torchvision import models

FINAL = False   # False: trial run (1 seed, scores on the validation split). True: final run (3 seeds, test split)

SEEDS = [0, 1, 2] if FINAL else [0]
RUN = 'final' if FINAL else 'trial'   # the output files start with 02_trial_ or 02_final_
EVAL = 'test' if FINAL else 'val'     # the split the scores are measured on
MODELS = ['mobilenet_v2', 'resnet50', 'custom_cnn', 'vgg16']   # the fast ones first
NAMES = {'custom_cnn': 'Custom CNN', 'vgg16': 'VGG16', 'resnet50': 'ResNet50', 'mobilenet_v2': 'MobileNetV2'}
APP_MODEL = 'mobilenet_v2'            # also saved as models/model.pt for the web app

# Training settings: the same for every model, and later for every row of the comparison table (notebook 04)
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30                  # at most
PATIENCE = 5                 # stop when the validation macro-F1 has not improved for this many epochs
LR = {'custom_cnn': 1e-3, 'vgg16': 1e-4, 'resnet50': 1e-4, 'mobilenet_v2': 1e-4}   # start learning rate
WEIGHT_DECAY = 0.01          # AdamW's default
PRETRAINED = True            # ImageNet weights for VGG16, ResNet50 and MobileNetV2
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet normalisation, as in the web app
EVAL_BATCH = 128             # batch size for scoring only: faster, same results

# The final split of notebook 01 (12,698 / 2,728 / 2,734 images) as a fingerprint of every image's path and split,
# so that an older version of the split cannot be used by mistake
SPLIT_FINGERPRINT = '7cf95c207559517a'

# Kaggle folders (the two environment variables are only for a test run on another computer)
INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
OUT_DIR = Path(os.environ.get('OUT_DIR', '/kaggle/working'))
RESULTS_OUT, MODELS_OUT = OUT_DIR / 'results', OUT_DIR / 'models'
RESULTS_OUT.mkdir(parents=True, exist_ok=True)
MODELS_OUT.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    try:   # a GPU this PyTorch version does not support fails at its first calculation
        nn.functional.conv2d(torch.ones(1, 1, 4, 4, device=DEVICE), torch.ones(1, 1, 3, 3, device=DEVICE))
    except RuntimeError as e:
        raise RuntimeError(f'The GPU {torch.cuda.get_device_name()} does not work with this PyTorch ({e}). '
                           'Choose Settings → Accelerator → GPU T4 x2 and run again.') from None
    torch.backends.cudnn.benchmark = True   # picks the fastest convolution method for the image size
    print('GPU:', torch.cuda.get_device_name(), '| PyTorch', torch.__version__)
elif Path('/kaggle').exists():
    raise RuntimeError('No GPU. Choose Settings → Accelerator → GPU T4 x2 and run again.')
else:
    DEVICE = torch.device('cpu')
    print('No GPU: running on the CPU (only for a short test run on another computer)')
AMP = DEVICE.type == 'cuda'   # mixed precision (16-bit numbers) on the GPU: about twice as fast
warnings.filterwarnings('ignore', message='Detected call of `lr_scheduler')   # harmless, in the first steps of AMP
print(f'{RUN} run: seeds {SEEDS}, scores on the {EVAL} split')

# Chart style of the project (as in notebook 01): light background, dark text, thin grey grid;
# blue = training, orange = validation
SURFACE, INK, INK_2, GRID, BLUE, ORANGE = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6', '#eb6834'
BLUES = LinearSegmentedColormap.from_list('blues', [SURFACE, '#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b'])
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': GRID, 'axes.linewidth': 1, 'grid.color': GRID, 'grid.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

## 1. The split and the images

`splits.csv` comes from notebook 01. Every image is read once, cropped to the centre square and resized to
224 × 224 (bilinear), the same way the web app prepares a photo, and kept in memory as 8-bit pixels (about 2.3 GB
for train and validation). Each epoch then reads from memory, which is much faster than opening the files again.
There is no augmentation in this phase.

In [ ]:
def find_files(root, name):
    """Every file called `name` under root. Class folders (names with ___) only hold images, so they are skipped."""
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        dirnames[:] = [d for d in dirnames if '___' not in d]
        if name in filenames:
            found.append(Path(dirpath) / name)
    return found

def find_color_dir(root):
    """PlantVillage's `color` folder (the one that holds the Tomato___ class folders) under root."""
    for dirpath, dirnames, _ in os.walk(root):
        dirnames.sort()
        if Path(dirpath).name.lower() == 'color' and any(d.startswith('Tomato') for d in dirnames):
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if '___' not in d]
    raise FileNotFoundError(f'No PlantVillage "color" folder under {root}. '
                            'Add the Kaggle dataset abdallahalidev/plantvillage-dataset as input.')

def fingerprint(splits):
    return hashlib.sha256('\n'.join(sorted(splits['path'] + ',' + splits['split'])).encode()).hexdigest()[:16]

found = {f: pd.read_csv(f) for f in find_files(INPUT_ROOT, 'splits.csv')}
final_split = [f for f, s in found.items() if fingerprint(s) == SPLIT_FINGERPRINT]
if not final_split:
    raise FileNotFoundError('The final splits.csv of notebook 01 was not found'
                            + (f' (only other versions: {[str(f) for f in found]})' if found else '')
                            + '. Add Input → Your Work → Notebooks → notebook 01, the version that made the '
                              '12,698 / 2,728 / 2,734 split.')
SPLITS_CSV = final_split[0]
splits = found[SPLITS_CSV]
classes = json.loads((SPLITS_CSV.parent / 'classes.json').read_text())
K = len(classes)
assert (splits['label'] == splits['class_folder'].map({c['folder']: c['label'] for c in classes})).all(), \
    'the labels in splits.csv do not match classes.json'
COLOR_DIR = find_color_dir(INPUT_ROOT)
print('Split: ', SPLITS_CSV)
print('Images:', COLOR_DIR)

# classes from the most to the fewest training images: the order of every table and chart below
train_counts = splits.loc[splits['split'] == 'train', 'label'].value_counts()
ORDER = [int(i) for i in train_counts.index]
CLASS_NAMES = [classes[i]['name'] for i in ORDER]
counts = pd.crosstab(splits['class_name'], splits['split']).loc[CLASS_NAMES, ['train', 'val', 'test']]
print(counts.to_string())

In [ ]:
def load_image(rel_path):
    """One image as IMG_SIZE x IMG_SIZE x 3 pixels (0-255): the centre square, resized bilinear, as in the web app."""
    with Image.open(COLOR_DIR / rel_path) as img:
        img = img.convert('RGB')
        side = min(img.size)
        left, top = (img.width - side) // 2, (img.height - side) // 2
        img = img.crop((left, top, left + side, top + side))
        return np.asarray(img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR))

def load_split(name):
    part = splits[splits['split'] == name]
    x = np.empty((len(part), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    with ThreadPoolExecutor(2 * (os.cpu_count() or 2)) as pool:
        for i, pixels in enumerate(pool.map(load_image, part['path'])):
            x[i] = pixels
    return torch.from_numpy(x), torch.tensor(part['label'].to_numpy())

start = time.time()
data = {name: load_split(name) for name in (['train', 'val', 'test'] if FINAL else ['train', 'val'])}
for name, (x, y) in data.items():
    print(f'{name:5s} {len(x):6,} images  {x.nbytes / 1e9:.2f} GB')
print(f'Read in {time.time() - start:.0f} s')

## 2. The four models

| Model | Start weights | Replaced by a new 10-class layer | Start learning rate |
|---|---|---|---|
| Custom CNN | random | (built here) | 0.001 |
| VGG16 | ImageNet (`IMAGENET1K_V1`) | `classifier[6]` | 0.0001 |
| ResNet50 | ImageNet (`IMAGENET1K_V2`) | `fc` | 0.0001 |
| MobileNetV2 | ImageNet (`IMAGENET1K_V2`) | `classifier[1]` | 0.0001 |

The **custom CNN** has 5 blocks; each block is two 3 × 3 convolutions (each followed by batch normalisation and ReLU)
and a 2 × 2 max pool, with 32, 64, 128, 256 and 512 channels. Then global average pooling, dropout (0.2) and the
10-class layer.

The three pretrained models are trained in full (all layers), starting from their ImageNet weights. The custom CNN
gets a 10 times larger learning rate because it starts from random weights: 0.0001 is a usual rate for fine-tuning
a pretrained network, and a network that starts from nothing would learn too slowly with it.

**The same for all four:** AdamW (weight decay 0.01), batch size 32, cross-entropy loss, up to 30 epochs, the
learning rate falling from its start value towards 0 along a cosine curve over the 30 epochs, 16-bit mixed precision
on the GPU. After every epoch the model is scored on the validation split. Training stops when the validation
macro-F1 has not improved for 5 epochs, and the best epoch is kept (on equal macro-F1, the lower validation loss
wins). No augmentation and no class weights: those come in notebook 04 (rows B and C).

In [ ]:
class CustomCNN(nn.Module):
    """5 blocks of [3x3 conv + batch norm + ReLU] x 2 + 2x2 max pool; then average pool, dropout, linear layer."""

    def __init__(self, num_classes, widths=(32, 64, 128, 256, 512)):
        super().__init__()
        layers, c_in = [], 3
        for c in widths:
            for _ in range(2):
                layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True)]
                c_in = c
            layers.append(nn.MaxPool2d(2))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2),
                                        nn.Linear(c_in, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(arch, pretrained):
    """The network with a new 10-class last layer, replaced the same way as in the web app (app/backend/model.py)."""
    if arch == 'custom_cnn':
        return CustomCNN(K)
    if arch == 'vgg16':
        net = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        net.classifier[6] = nn.Linear(net.classifier[6].in_features, K)
    elif arch == 'resnet50':
        net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        net.fc = nn.Linear(net.fc.in_features, K)
    elif arch == 'mobilenet_v2':
        net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2 if pretrained else None)
        net.classifier[1] = nn.Linear(net.last_channel, K)
    else:
        raise ValueError(f'unknown model {arch}')
    return net

# Build each model once now: this downloads the ImageNet weights (kept on disk for the runs) and counts the weights
params = {}
for arch in MODELS:
    try:
        net = build_model(arch, PRETRAINED)
    except Exception as e:
        raise RuntimeError(f'Could not build {arch}: {e}\nIf the ImageNet weights did not download: '
                           'Settings → Internet on, then run again.') from None
    params[arch] = sum(p.numel() for p in net.parameters())
    print(f'{NAMES[arch]:12s} {params[arch] / 1e6:6.1f} million parameters')
del net

## 3. Train

Each model is trained once per seed. The seed sets the starting weights of the new last layer (of all layers, for
the custom CNN) and the order of the training images. Every epoch prints one line: the training loss and accuracy,
then the validation loss, accuracy and macro-F1; `best` marks a new best epoch.

After each run, the best run of each model so far (highest validation macro-F1, never judged on test) is saved as
`models/<model>.pt`. MobileNetV2 is also saved as `models/model.pt` for the web app. The results so far are written
to `results/` after every run, so a run that is cut off still leaves the finished ones.

In [ ]:
MEAN_255 = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1) * 255
STD_255 = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1) * 255

def to_input(pixels):
    """8-bit images (n, H, W, 3) -> normalised float tensor (n, 3, H, W) on the GPU, in channels-last memory layout."""
    x = pixels.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float()
    return (x - MEAN_255) / STD_255

def autocast():
    return torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP)

def scores(y_true, y_pred):
    """Accuracy, precision / recall / F1 per class and their macro averages (the plain mean over the 10 classes)."""
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(K), zero_division=0)
    return {'accuracy': float(np.mean(y_true == y_pred)), 'macro_precision': float(p.mean()),
            'macro_recall': float(r.mean()), 'macro_f1': float(f1.mean()),
            'precision': p.tolist(), 'recall': r.tolist(), 'f1': f1.tolist(),
            'confusion': confusion_matrix(y_true, y_pred, labels=range(K)).tolist()}   # rows: true class, by label

@torch.inference_mode()
def evaluate(net, split):
    """Mean loss and scores on a whole split."""
    x, y = data[split]
    net.eval()
    loss, preds = 0.0, []
    for i in range(0, len(x), EVAL_BATCH):
        with autocast():
            logits = net(to_input(x[i:i + EVAL_BATCH])).float()
        loss += nn.functional.cross_entropy(logits, y[i:i + EVAL_BATCH].to(DEVICE), reduction='sum').item()
        preds.append(logits.argmax(1).cpu())
    return loss / len(x), scores(y.numpy(), torch.cat(preds).numpy())

def train_epoch(net, opt, sched, scaler, gen):
    x, y = data['train']
    net.train()
    total_loss = torch.zeros((), device=DEVICE)
    correct = torch.zeros((), device=DEVICE)
    for idx in torch.randperm(len(x), generator=gen).split(BATCH_SIZE):
        xb, yb = to_input(x[idx]), y[idx].to(DEVICE, non_blocking=True)
        with autocast():
            logits = net(xb)
            loss = nn.functional.cross_entropy(logits, yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total_loss += loss.detach().float() * len(idx)
        correct += (logits.argmax(1) == yb).sum()
    return total_loss.item() / len(x), correct.item() / len(x)

def better(a, b):
    """Is epoch a better than epoch b? Higher validation macro-F1; on equal macro-F1, lower validation loss."""
    return b is None or (a['val_macro_f1'], -a['val_loss']) > (b['val_macro_f1'], -b['val_loss'])

def train_run(arch, seed):
    """Train one model with one seed. Returns the network with its best epoch's weights, the history and that epoch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    net = build_model(arch, PRETRAINED).to(DEVICE, memory_format=torch.channels_last)
    opt = torch.optim.AdamW(net.parameters(), lr=LR[arch], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(data['train'][0]) / BATCH_SIZE)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * steps_per_epoch)   # stepped every batch
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    gen = torch.Generator().manual_seed(seed)
    history, best, best_weights = [], None, None
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        lr = opt.param_groups[0]['lr']
        train_loss, train_acc = train_epoch(net, opt, sched, scaler, gen)
        val_loss, val = evaluate(net, 'val')
        row = {'epoch': epoch, 'lr': lr, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss,
               'val_acc': val['accuracy'], 'val_macro_f1': val['macro_f1'], 'seconds': round(time.time() - start, 1)}
        history.append(row)
        is_best = better(row, best)
        if is_best:
            best = row
            best_weights = {k: v.detach().to('cpu', copy=True) for k, v in net.state_dict().items()}
        print(f'epoch {epoch:2d}/{EPOCHS}  train loss {train_loss:.4f} acc {100 * train_acc:6.2f}%  |  '
              f'val loss {val_loss:.4f} acc {100 * val["accuracy"]:6.2f}% macro-F1 {100 * val["macro_f1"]:6.2f}%  |  '
              f'{row["seconds"]:4.0f} s{"  best" if is_best else ""}')
        if epoch - best['epoch'] >= PATIENCE:
            print(f'Stopped early: no better validation macro-F1 for {PATIENCE} epochs')
            break
    net.load_state_dict(best_weights)
    return net, history, best

In [ ]:
def save_model(net, arch, run, path):
    """Save in the format the web app loads (app/backend/model.py), plus the run's details for notebooks 03 and 04."""
    metrics = {'validation_accuracy': run['val']['accuracy'], 'validation_macro_F1': run['val']['macro_f1']}
    if 'test' in run:
        metrics.update(test_accuracy=run['test']['accuracy'], test_macro_F1=run['test']['macro_f1'])
    torch.save({
        'arch': arch,
        'classes': classes,
        'state_dict': {k: v.detach().cpu().contiguous() for k, v in net.state_dict().items()},
        'img_size': IMG_SIZE,
        'mean': MEAN,
        'std': STD,
        'name': f'{NAMES[arch]} (real images only)',
        'metrics': metrics,
        'seed': run['seed'],
        'best_epoch': run['best_epoch'],
        'run': RUN,
    }, path)

settings = {
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'run': RUN, 'seeds': SEEDS, 'scores_on': EVAL, 'models': MODELS, 'training_images': 'real only (train split)',
    'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS, 'patience': PATIENCE,
    'early_stopping': 'validation macro-F1; on equal macro-F1, lower validation loss',
    'optimizer': f'AdamW, weight decay {WEIGHT_DECAY}', 'start_lr': LR,
    'lr_schedule': 'cosine from the start lr to 0 over max_epochs, stepped every batch',
    'loss': 'cross-entropy, no class weights', 'augmentation': 'none',
    'pretrained': ('ImageNet: VGG16 IMAGENET1K_V1, ResNet50 IMAGENET1K_V2, MobileNetV2 IMAGENET1K_V2'
                   if PRETRAINED else 'none'),
    'normalisation': {'mean': MEAN, 'std': STD}, 'mixed_precision_fp16': AMP,
    'split_csv': str(SPLITS_CSV), 'split_fingerprint': SPLIT_FINGERPRINT,
    'device': torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'CPU', 'torch': torch.__version__,
}
RUNS_JSON = RESULTS_OUT / f'02_{RUN}_runs.json'
key = lambda run: (run['val']['macro_f1'], -run['val_loss'])   # which run of a model is saved
runs, saved = [], {}   # saved: model -> the run whose weights are in models/<model>.pt

start_all = time.time()
for arch in MODELS:
    for seed in SEEDS:
        print(f'\n=== {NAMES[arch]}, seed {seed} ===')
        start = time.time()
        net, history, best = train_run(arch, seed)
        run = {'model': arch, 'seed': seed, 'best_epoch': best['epoch'], 'epochs_run': len(history),
               'minutes': round((time.time() - start) / 60, 1), 'history': history}
        run['val_loss'], run['val'] = evaluate(net, 'val')
        if FINAL:
            run['test_loss'], run['test'] = evaluate(net, 'test')
        runs.append(run)
        if arch not in saved or key(run) > key(saved[arch]):
            saved[arch] = run
            save_model(net, arch, run, MODELS_OUT / f'{arch}.pt')
            if arch == APP_MODEL:
                shutil.copy(MODELS_OUT / f'{arch}.pt', MODELS_OUT / 'model.pt')
        RUNS_JSON.write_text(json.dumps({'settings': settings, 'runs': runs}))
        print(f'Best epoch {run["best_epoch"]} of {run["epochs_run"]}: validation macro-F1 '
              f'{100 * run["val"]["macro_f1"]:.2f}%, {run["minutes"]} minutes')
        del net
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
print(f'\nAll {len(runs)} runs done in {(time.time() - start_all) / 3600:.1f} hours')

## 4. Size and prediction time

For the saved model of each kind: the number of parameters, the file size, and the time to classify one photo
(batch of 1, 32-bit numbers, the median of many repeats) on the GPU and on Kaggle's CPU. The web app runs on a CPU,
so the CPU time is the one that matters for it; the laptop or the hosting server will be somewhat faster or slower.

In [ ]:
def ms_per_image(net, device, repeats):
    net = net.to(device).eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    wait = torch.cuda.synchronize if device.type == 'cuda' else (lambda: None)
    times = []
    with torch.inference_mode():
        for i in range(repeats + 5):   # the first 5 are a warm-up and not counted
            wait()
            start = time.perf_counter()
            net(x)
            wait()
            if i >= 5:
                times.append(time.perf_counter() - start)
    return 1000 * float(np.median(times))

speed = {}
for arch in MODELS:
    path = MODELS_OUT / f'{arch}.pt'
    net = build_model(arch, pretrained=False)
    net.load_state_dict(torch.load(path, map_location='cpu', weights_only=True)['state_dict'])
    speed[arch] = {
        'params_million': round(params[arch] / 1e6, 2),
        'size_mb': round(path.stat().st_size / 1e6, 1),
        'gpu_ms': round(ms_per_image(net, DEVICE, 50), 1) if DEVICE.type == 'cuda' else None,
        'cpu_ms': round(ms_per_image(net, torch.device('cpu'), 20), 1),
    }
    s = speed[arch]
    print(f'{NAMES[arch]:12s} {s["params_million"]:6.1f} M parameters  {s["size_mb"]:6.1f} MB   '
          f'GPU {s["gpu_ms"] if s["gpu_ms"] is not None else "-":>6} ms   CPU {s["cpu_ms"]:7.1f} ms per image')
    del net
print(f'(CPU: {os.cpu_count()} cores)')

## 5. Results

Trial run: scores on the validation split (1 seed). Final run: scores on the test split, the mean ± standard deviation
over the 3 seeds. The confusion matrices are those of the saved model of each kind.

In [ ]:
def pct(values):
    """Mean ± standard deviation in %, or the single value when there is one seed."""
    v = 100 * np.asarray(values, dtype=float)
    return f'{v.mean():.2f} ± {v.std(ddof=1):.2f}' if len(v) > 1 else f'{v[0]:.2f}'

SPLIT_NAME = 'test' if FINAL else 'validation'
runs_of = {arch: [r for r in runs if r['model'] == arch] for arch in MODELS}

results = pd.DataFrame([{
    'Model': NAMES[arch],
    'Accuracy %': pct([r[EVAL]['accuracy'] for r in rs]),
    'Macro precision %': pct([r[EVAL]['macro_precision'] for r in rs]),
    'Macro recall %': pct([r[EVAL]['macro_recall'] for r in rs]),
    'Macro-F1 %': pct([r[EVAL]['macro_f1'] for r in rs]),
    'Best epoch': ', '.join(str(r['best_epoch']) for r in rs),
    'Parameters (M)': round(speed[arch]['params_million'], 1),
    'File size (MB)': speed[arch]['size_mb'],
    'CPU ms / image': speed[arch]['cpu_ms'],
    'GPU ms / image': speed[arch]['gpu_ms'],
    'Training min / seed': round(float(np.mean([r['minutes'] for r in rs])), 1),
} for arch, rs in runs_of.items()])
results.to_csv(RESULTS_OUT / f'02_{RUN}_results.csv', index=False)
print(f'Scores on the {SPLIT_NAME} split, {len(SEEDS)} seed{"s" if len(SEEDS) > 1 else ""}:\n')
print(results.to_string(index=False))

# one row per model and seed, plain numbers (for notebook 04 and for checking)
pd.DataFrame([{'model': NAMES[r['model']], 'seed': r['seed'], 'best_epoch': r['best_epoch'],
               'epochs_run': r['epochs_run'], 'minutes': r['minutes'],
               **{f'{EVAL}_{m}': r[EVAL][m] for m in ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1')}}
              for r in runs]).to_csv(RESULTS_OUT / f'02_{RUN}_runs.csv', index=False)

per_class = pd.DataFrame([{'model': NAMES[arch], 'class': classes[i]['name'], 'train images': int(train_counts[i]),
                           **{f'{m} %': pct([r[EVAL][m][i] for r in rs]) for m in ('precision', 'recall', 'f1')}}
                          for arch, rs in runs_of.items() for i in ORDER])
per_class.to_csv(RESULTS_OUT / f'02_{RUN}_per_class.csv', index=False)

recall = pd.DataFrame({NAMES[arch]: [100 * np.mean([r[EVAL]['recall'][i] for r in rs]) for i in ORDER]
                       for arch, rs in runs_of.items()}, index=CLASS_NAMES)
print(f'\nRecall per class (% of the class\'s {SPLIT_NAME} images that were found), mean over the seeds:\n')
print(recall.round(2).to_string())
print('\nWeakest classes (lowest recall) of each model:')
for arch in MODELS:
    weakest = recall[NAMES[arch]].nsmallest(3)
    print(f'  {NAMES[arch]:12s} ' + ', '.join(f'{c} {v:.1f}%' for c, v in weakest.items()))

In [ ]:
def plain_log_ticks(ax):
    """Plain numbers (0.01, 0.1, 1) on a log axis, plus 0.02, 0.05 ... when the axis spans less than 2 powers of 10."""
    lo, hi = ax.get_ylim()
    between = math.log10(hi / lo) < 2
    mantissa = lambda v: round(v / 10 ** math.floor(math.log10(v)), 6)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: f'{v:g}' if between and mantissa(v) in (2, 5) else ''))

fig, axes = plt.subplots(2, len(MODELS), figsize=(3.6 * len(MODELS), 6.6), squeeze=False)
for col, (arch, rs) in enumerate(runs_of.items()):
    top, bottom = axes[0, col], axes[1, col]
    for r in rs:
        h = pd.DataFrame(r['history'])
        top.plot(h['epoch'], h['train_loss'], color=BLUE, linewidth=1.5)
        top.plot(h['epoch'], h['val_loss'], color=ORANGE, linewidth=1.5)
        bottom.plot(h['epoch'], 100 * h['val_macro_f1'], color=ORANGE, linewidth=1.5)
        b = h[h['epoch'] == r['best_epoch']].iloc[0]
        bottom.plot(b['epoch'], 100 * b['val_macro_f1'], 'o', color=ORANGE, markersize=7,
                    markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=3)
    top.set_yscale('log')
    plain_log_ticks(top)
    top.set_title(NAMES[arch])
    for ax in (top, bottom):
        ax.yaxis.grid(True)
        ax.set_axisbelow(True)
        ax.set_xlim(0.5, max(len(r['history']) for r in rs) + 0.5)
        ax.xaxis.get_major_locator().set_params(integer=True)
    bottom.set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (log scale)')
axes[1, 0].set_ylabel('Validation macro-F1 %')
fig.legend([Line2D([], [], color=BLUE, linewidth=1.5), Line2D([], [], color=ORANGE, linewidth=1.5),
            Line2D([], [], color=ORANGE, marker='o', linestyle='', markersize=7, markeredgecolor=SURFACE)],
           ['training', 'validation', 'best epoch (kept)'], loc='upper right', ncol=3, frameon=False,
           bbox_to_anchor=(1, 1.0))
seeds_note = f'{len(SEEDS)} seeds, one line each' if len(SEEDS) > 1 else 'seed ' + str(SEEDS[0])
fig.suptitle(f'Training curves ({seeds_note})', x=0.01, y=1.0, ha='left', fontweight='bold', fontsize=13)
fig.tight_layout()   # leaves room for the title row
fig.savefig(RESULTS_OUT / f'02_{RUN}_training_curves.png')
plt.show()

In [ ]:
def cell_grid(ax, rows, cols):
    """A 2 px gap in the background colour between the cells of a heatmap."""
    ax.set_xticks(np.arange(cols + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(rows + 1) - 0.5, minor=True)
    ax.grid(which='minor', color=SURFACE, linewidth=2)
    ax.grid(which='major', visible=False)
    ax.tick_params(which='both', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

for arch in MODELS:
    run = saved[arch]
    cm = np.array(run[EVAL]['confusion'])[np.ix_(ORDER, ORDER)]   # classes from the most to the fewest images
    share = cm / np.maximum(cm.sum(1, keepdims=True), 1)          # share of each true class
    fig, ax = plt.subplots(figsize=(8.4, 7.2))
    im = ax.imshow(share, cmap=BLUES, norm=PowerNorm(0.5, vmin=0, vmax=1))   # square-root scale: small mistakes show
    for i in range(K):
        for j in range(K):
            if cm[i, j]:
                ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=9,
                        color='white' if share[i, j] > 0.3 else INK)
    ax.set_xticks(range(K), CLASS_NAMES, rotation=40, ha='right', rotation_mode='anchor')
    ax.set_yticks(range(K), CLASS_NAMES)
    cell_grid(ax, K, K)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_title(f'{NAMES[arch]}: confusion matrix, {SPLIT_NAME} split', pad=24)
    ax.text(0, 1.015, f'Seed {run["seed"]}: accuracy {100 * run[EVAL]["accuracy"]:.2f}%, macro-F1 '
            f'{100 * run[EVAL]["macro_f1"]:.2f}%. Numbers: images (empty = 0).',
            transform=ax.transAxes, color=INK_2, fontsize=9)
    ticks = [0, 0.01, 0.05, 0.2, 0.5, 1]
    bar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, ticks=ticks)
    bar.ax.set_yticklabels([f'{100 * t:g}%' for t in ticks])
    bar.outline.set_visible(False)
    bar.set_label('Share of the true class')
    fig.savefig(RESULTS_OUT / f'02_{RUN}_confusion_{arch}.png')
    plt.show()

values = recall.to_numpy()
low = min(90.0, 5 * math.floor(values.min() / 5))
fig, ax = plt.subplots(figsize=(2.6 + 1.5 * len(MODELS), 6))
im = ax.imshow(values, cmap=BLUES, vmin=low, vmax=100, aspect='auto')
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        ax.text(j, i, f'{values[i, j]:.1f}', ha='center', va='center', fontsize=10,
                color='white' if (values[i, j] - low) / (100 - low) > 0.55 else INK)
ax.set_xticks(range(len(MODELS)), recall.columns)
ax.set_yticks(range(K), [f'{name}  ({train_counts[i]:,})' for name, i in zip(CLASS_NAMES, ORDER)])
cell_grid(ax, K, len(MODELS))
ax.set_title(f'Recall per class (%), {SPLIT_NAME} split', pad=24)
ax.text(0, 1.015, 'Brackets: training images of the class. '
        + (f'Mean of {len(SEEDS)} seeds.' if len(SEEDS) > 1 else f'Seed {SEEDS[0]}.'),
        transform=ax.transAxes, color=INK_2, fontsize=9)
fig.savefig(RESULTS_OUT / f'02_{RUN}_per_class_recall.png')
plt.show()

## 6. Save

In [ ]:
RUNS_JSON.write_text(json.dumps({
    'settings': settings,
    'runs': runs,
    'speed': speed,
    'saved_models': {arch: {'file': f'models/{arch}.pt', 'seed': saved[arch]['seed']} for arch in MODELS},
    'app_model': {'file': 'models/model.pt', 'arch': APP_MODEL},
}, indent=1))

if FINAL:   # the rows for the About page of the web app
    print('For app/frontend/src/data/results.js:\n')
    for arch, rs in runs_of.items():
        print(f"  {{ model: '{NAMES[arch]}', images: 'Real only', "
              f"accuracy: {np.mean([r['test']['accuracy'] for r in rs]):.4f}, "
              f"macroF1: {np.mean([r['test']['macro_f1'] for r in rs]):.4f} }},")
    print()

for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file() and f.parent.name in ('results', 'models') and (f.parent.name == 'models' or f.name.startswith('02_')):
        print(f'{f.relative_to(OUT_DIR)}  ({f.stat().st_size / 1e6:,.1f} MB)')

## Checklist

1. **Training curves:** both losses fall and then level off. A validation loss that rises again while the training
   loss keeps falling is overfitting; early stopping keeps the epoch before it. A loss that does not fall at all, or
   jumps up and down, means the learning rate is too high. The custom CNN is expected to learn more slowly.
2. **Results table:** the pretrained models are expected at about 97–99% accuracy, the custom CNN lower. PlantVillage
   is an easy dataset, so compare the models on macro-F1 and on the recall per class, not only on accuracy.
3. **Recall per class:** note the weakest classes. The GAN (notebook 03) is meant to help the small ones: Mosaic
   virus, Leaf mold and Early blight.
4. **Confusion matrices:** which classes are mixed up with which. These pairs go in the report.
5. **Best epoch:** if a model's best epoch is the last one (30), it was still improving when training ended. Note it.

If the trial run looks fine, set `FINAL = True` and commit again. After the final run, download `results/` and
`models/model.pt` into the project folder: the web app loads `models/model.pt` the next time `run-app.bat` starts, and
the "Demo mode" banner goes away. The printed `results.js` rows go into `app/frontend/src/data/results.js`.

Next is notebook 03 (the GAN), trained on the train split only.